# Notebook 10.1 — Assistant Backend & Hybrid Query Router

## Objective
I turn the Phase 9 retrieval pipeline into a reusable RetainIQ query engine. I keep RAG for contextual knowledge and add a small, explicit SQL-routing layer for exact portfolio metrics and comparison questions.

## Notebook presentation rule
For relevant analytical cells I place an immediate **Result & conclusion** cell after the code. Setup-only cells are not padded with artificial conclusions.

In [1]:
import sys
from pathlib import Path
import pandas as pd

APP_DIR = Path("../app").resolve() if (Path("../app") / "query_engine.py").exists() else Path("app").resolve()
sys.path.insert(0, str(APP_DIR))

from query_engine import (
    CHUNKS_DF, VECTORS, MANIFEST, PARAMETERS,
    is_comparison_query, retrieve, build_grounded_context,
    answer_query
)

print_result = lambda message: print(f"Result: {message}")
print_result(f"Loaded {len(CHUNKS_DF):,} retrieval chunks using the {MANIFEST.get('backend', 'unknown')} backend.")


Result: Loaded 8,414 retrieval chunks using the tfidf backend.


### Result & conclusion
I confirmed that the Phase 9 retrieval index is available to the Phase 10 backend, so I can reuse the tested knowledge layer rather than rebuilding it.

In [2]:
test_queries = pd.DataFrame({
    'question': [
        'What is the definition of revenue at risk?',
        'Which customer segment has the highest revenue at risk?',
        'How does a city compare with its peer-city benchmark?',
        'What does a 20 percent recovery scenario mean?'
    ]
})

test_queries['comparison_query'] = test_queries['question'].map(is_comparison_query)
display(test_queries)
print_result(f'{int(test_queries["comparison_query"].sum())} of {len(test_queries)} test questions are routed as comparison-style questions.')

,question,comparison_query
0,What is the definition of revenue at risk?,False
1,Which customer segment has the highest revenue...,True
2,How does a city compare with its peer-city ben...,True
3,What does a 20 percent recovery scenario mean?,False


Result: 2 of 4 test questions are routed as comparison-style questions.


### Result & conclusion
The router separates comparison-style questions from definition/context questions using explicit language markers. This is a lightweight routing rule, not a machine-learning classifier.

In [3]:
for q in test_queries['question']:
    results = retrieve(q, top_k=5)
    print('=' * 90)
    print(q)
    display(results[['evidence_id','source_name','topic','similarity','chunk_text']])


What is the definition of revenue at risk?


,evidence_id,source_name,topic,similarity,chunk_text
0,[1],project_methodology,definitions,0.145832,GenAI / RAG Knowledge Layer ## Core definition...
1,[2],vw_retention_strategy_priority,retention_strategy,0.144830,Source: vw_retention_strategy_priority | Topic...
2,[3],vw_benchmark_reference,benchmarking,0.138585,Source: vw_benchmark_reference | Topic: benchm...
3,[4],vw_retention_strategy_priority,retention_strategy,0.133413,Source: vw_retention_strategy_priority | Topic...
4,[5],vw_retention_strategy_priority,retention_strategy,0.131462,Source: vw_retention_strategy_priority | Topic...


Which customer segment has the highest revenue at risk?


,evidence_id,source_name,topic,similarity,chunk_text
0,[1],vw_benchmark_reference,benchmarking,0.232903,Source: vw_benchmark_reference | Topic: benchm...
1,[2],vw_benchmark_reference,benchmarking,0.165717,Source: vw_benchmark_reference | Topic: benchm...
2,[3],vw_city_benchmark_management,benchmarking,0.153579,Source: vw_city_benchmark_management | Topic: ...
3,[4],vw_city_benchmark_management,benchmarking,0.153383,Source: vw_city_benchmark_management | Topic: ...
4,[5],vw_retention_impact_scenarios,retention_strategy,0.151980,Source: vw_retention_impact_scenarios | Topic:...


How does a city compare with its peer-city benchmark?


,evidence_id,source_name,topic,similarity,chunk_text
0,[1],project_methodology,definitions,0.302627,chmark gap `Benchmark Gap = Observed Churn Rat...
1,[2],vw_city_benchmark_management,benchmarking,0.103970,Source: vw_city_benchmark_management | Topic: ...
2,[3],vw_city_benchmark_management,benchmarking,0.102348,Source: vw_city_benchmark_management | Topic: ...


What does a 20 percent recovery scenario mean?


,evidence_id,source_name,topic,similarity,chunk_text
0,[1],vw_retention_action_framework,retention_strategy,0.140322,Source: vw_retention_action_framework | Topic:...
1,[2],project_methodology,definitions,0.138446,GenAI / RAG Knowledge Layer ## Core definition...


### Result & conclusion
I can now inspect the exact evidence returned for each question. For comparison questions, the backend considers a wider candidate pool before selecting diverse source records.

In [4]:
comparison_q = "Which customer segment has the highest revenue at risk?"
result = answer_query(comparison_q, mysql_password=None, top_k=5)

print_result(f"Mode = {result['mode']}; retrieved {len(result['retrieved'])} semantic evidence items.")
print("\nAnswer preview:\n", result["answer"][:1500])
print("\nEvidence:")
display(result["retrieved"][["evidence_id","source_name","topic","similarity","chunk_text"]])


Result: Mode = llm; retrieved 5 semantic evidence items.

Answer preview:
 Segment 0 — Emerging Risk (evidence [5])

Evidence:


,evidence_id,source_name,topic,similarity,chunk_text
0,[1],vw_benchmark_reference,benchmarking,0.232903,Source: vw_benchmark_reference | Topic: benchm...
1,[2],vw_benchmark_reference,benchmarking,0.165717,Source: vw_benchmark_reference | Topic: benchm...
2,[3],vw_city_benchmark_management,benchmarking,0.153579,Source: vw_city_benchmark_management | Topic: ...
3,[4],vw_city_benchmark_management,benchmarking,0.153383,Source: vw_city_benchmark_management | Topic: ...
4,[5],vw_retention_impact_scenarios,retention_strategy,0.151980,Source: vw_retention_impact_scenarios | Topic:...


### Result & conclusion
The hybrid engine can produce a grounded answer even when MySQL credentials are not supplied. When MySQL is available, exact portfolio/comparison evidence can be added before generation.

## Handoff
The reusable engine is now the foundation for the Streamlit assistant. The next notebook focuses on the comparison-completeness problem identified in Phase 9.